<a href="https://colab.research.google.com/github/detektor777/colab_list_video/blob/main/nafnet_video_continue_new_buffer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://drive.google.com/drive/my-drive



In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
!nvidia-smi
!git clone https://github.com/megvii-research/NAFNet.git
%cd NAFNet

# Install dependencies
!pip install -r requirements.txt
!pip install --upgrade --no-cache-dir gdown
!python3 setup.py develop --no_cuda_ext

# Download the pre-trained model
import gdown
import os

if not os.path.exists("./experiments/pretrained_models/NAFNet-REDS-width64.pth"):
  gdown.download('https://drive.google.com/uc?id=14D4V4raNYIOhETfcuuLI3bGLB-OYIv6X', "./experiments/pretrained_models/", quiet=False)

print("Installation and setup complete.")
%cd /content/NAFNet

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import glob
from google.colab import files
from tqdm.notebook import tqdm

from basicsr.models import create_model
from basicsr.utils import img2tensor as _img2tensor, tensor2img, imwrite
from basicsr.utils.options import parse

def imread(img_path):
  """Read image from path and convert to RGB."""
  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  return img

def img2tensor(img, bgr2rgb=False, float32=True):
    """Convert image to tensor."""
    img = img.astype(np.float32) / 255.
    return _img2tensor(img, bgr2rgb=bgr2rgb, float32=float32)

def single_image_inference(model, img, save_path):
      """Run inference on a single image."""
      model.feed_data(data={'lq': img.unsqueeze(dim=0)})

      if model.opt['val'].get('grids', False):
          model.grids()

      model.test()

      if model.opt['val'].get('grids', False):
          model.grids_inverse()

      visuals = model.get_current_visuals()
      sr_img = tensor2img([visuals['result']])
      imwrite(sr_img, save_path)

print("Libraries imported and helper functions defined.")

In [ ]:
#@title ##**Select Video File** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

upload_option = "Load from Google Drive Root"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive (includes subfolders)"]

file_name = None
last_selected_button = None

def reset_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

if upload_option == "Upload from PC":
    print("Please upload a video file.")
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
        # Move uploaded file to a standard path
        shutil.move(file_name, f'/content/{file_name}')
        file_name = f'/content/{file_name}'
    else:
        print("No file uploaded.")
        file_name = None

else:
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    if upload_option == "Load from Google Drive Root":
        for f in os.listdir(root_dir):
            if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in video_extensions:
                files_list.append(f)
    else: # Load from all subfolders
        for dirpath, _, filenames in os.walk(root_dir):
            for f in filenames:
                if os.path.splitext(f)[1].lower() in video_extensions:
                    relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                    files_list.append(relative_path)

    if not files_list:
        print("No video files found in the specified Google Drive location.")
        file_name = None
    else:
        print("Select a video file from Google Drive:")
        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'lightgreen'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='auto'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if file_name:
    print(f"Video file path set to: {file_name}")
else:
    print("Video file path not set. Please select a file.")

In [ ]:
#@title ##**Config** { display-mode: "form" }
import os
import shutil
from google.colab import drive

# --- Folder Settings ---
output_folder_location = "google_drive" #@param ["google_drive", "colab_runtime"]
clear_input_folder = False #@param {type:"boolean"}
clear_output_folder = False #@param {type:"boolean"}

# Define input and output paths based on selection
if output_folder_location == "google_drive":
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
    # Paths are directly in MyDrive, NOT in a 'NAFNet' subfolder
    base_drive_path = '/content/drive/MyDrive'
    input_folder = os.path.join(base_drive_path, 'deblur_input')
    output_folder = os.path.join(base_drive_path, 'deblur_output')
else:
    # Use local colab storage if selected
    input_folder = '/content/deblur_input'
    output_folder = '/content/deblur_output'

# Create folders if they don't exist
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# Handle folder clearing
if clear_input_folder:
    if os.path.isdir(input_folder):
        shutil.rmtree(input_folder)
    os.makedirs(input_folder)
    print(f"Input folder cleared: {input_folder}")

if clear_output_folder:
    if os.path.isdir(output_folder):
        shutil.rmtree(output_folder)
    os.makedirs(output_folder)
    print(f"Output folder cleared: {output_folder}")

print(f"\nInput frames will be stored in: {input_folder}")
print(f"Output frames will be stored in: {output_folder}")

In [ ]:
#@title ##**Run Sequence (Video to Frames)** { display-mode: "form" }
import cv2
import imageio
import os
import tqdm
import subprocess
import numpy as np
import time
from PIL import Image

if 'file_name' not in locals() or not file_name or not os.path.exists(file_name):
    raise ValueError("Video file not selected or does not exist. Please run the 'Select Video File' cell first.")
if 'input_folder' not in locals() or not input_folder:
    raise ValueError("Input folder is not defined. Please run the 'Config' cell.")

full_path = file_name

library = "ffmpeg" #@param ["cv2", "pyav", "imageio", "ffmpeg", "skvideo", "scipy", "moviepy"]
delay = "0.1" #@param ["0.0", "0.01", "0.05", "0.1", "0.2"]
delay_seconds = float(delay)

if (library == "cv2"):
    vidcap = cv2.VideoCapture(full_path)
    frame_count = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    print(f"FPS: {fps}, Frames: {frame_count}")
    pbar_cv2 = tqdm.tqdm(total=frame_count, desc="Extracting with CV2")
    for count in range(frame_count):
        frame_path = os.path.join(input_folder, f"frame{count+1:09d}.jpg")
        if os.path.exists(frame_path):
            pbar_cv2.update(1)
            continue
        success, image = vidcap.read()
        if not success: break
        cv2.imwrite(frame_path, image) # CV2 uses BGR, which is correct here
        pbar_cv2.update(1)
        if delay_seconds > 0: time.sleep(delay_seconds)
    pbar_cv2.close()
    vidcap.release()

elif (library == "ffmpeg"):
    import ffmpeg
    probe = ffmpeg.probe(full_path)
    video_info = next(s for s in probe['streams'] if s['codec_type'] == 'video')
    frame_count = int(video_info['nb_frames'])
    print(f"Frames: {frame_count}")
    width, height = int(video_info['width']), int(video_info['height'])

    pbar_ffmpeg = tqdm.tqdm(total=frame_count, desc="Extracting with FFMPEG")
    process = (ffmpeg.input(full_path).output('pipe:', format='rawvideo', pix_fmt='rgb24').run_async(pipe_stdout=True))
    for i in range(frame_count):
        frame_path = os.path.join(input_folder, f"frame{i+1:09d}.jpg")
        if os.path.exists(frame_path):
            process.stdout.read(width * height * 3) # Must read to advance stream
            pbar_ffmpeg.update(1)
            continue
        in_bytes = process.stdout.read(width * height * 3)
        if not in_bytes: break
        frame = np.frombuffer(in_bytes, np.uint8).reshape([height, width, 3])
        # FFMPEG outputs RGB, imageio saves RGB. NO cvtColor NEEDED. THIS FIXES THE COLOR INVERSION.
        imageio.imwrite(frame_path, frame)
        pbar_ffmpeg.update(1)
        if delay_seconds > 0: time.sleep(delay_seconds)
    pbar_ffmpeg.close()
    process.wait()

# ... (other libraries can be added here in the same fashion if needed) ...

# --- Final Check for Missing Frames ---
def check_frames_exist(folder):
    print("\nVerifying extracted frames...")
    try:
        frames = [int(re.findall(r'(\d+)', f)[0]) for f in os.listdir(folder) if f.endswith('.jpg')]
        if not frames:
            print("No frames found to verify.")
            return
        min_f, max_f = min(frames), max(frames)
        expected = set(range(min_f, max_f + 1))
        found = set(frames)
        missing = sorted(list(expected - found))
        if missing:
            print(f"Warning: {len(missing)} missing frames found. First few: {missing[:10]}")
        else:
            print("All frames are present.")
    except Exception as e:
        print(f"Could not verify frames due to an error: {e}")

check_frames_exist(input_folder)

In [ ]:
#@title ##**Run Deblur** { display-mode: "form" }
import shutil
from tqdm import tqdm
import os
import re
import cv2
import torch
import glob
import time

# --- Ensure folder paths are defined ---
if 'input_folder' not in locals() or 'output_folder' not in locals():
    raise NameError("Folder paths are not defined. Please run the 'Config' cell first.")

# --- 1. Pre-run Check of Input Frames ---
def check_frames(frame_dir, check_type="input"):
    print(f"\nVerifying {check_type} frames...")
    try:
        # Tries to find numbers in filenames like 'frame001.jpg', '001.jpg' etc.
        frames = [int(re.findall(r'(\d+)', f)[0]) for f in os.listdir(frame_dir) if f.endswith('.jpg') and re.findall(r'(\d+)', f)]
        if not frames:
            print(f"No frames found in the {check_type} directory to verify.")
            return True # Continue if empty, no error

        min_frame, max_frame = min(frames), max(frames)
        print(f"Frame range found: {min_frame} to {max_frame}")

        expected_frames = set(range(min_frame, max_frame + 1))
        actual_frames = set(frames)
        missing = sorted(list(expected_frames - actual_frames))

        if missing:
            print(f"Warning: Found {len(missing)} missing frames. First few are: {missing[:10]}")
        else:
            print("All frames appear to be present.")
        return True
    except Exception as e:
        print(f"Could not verify frames due to an error: {e}")
        return False

attempts = 0
max_attempts = 5
while attempts < max_attempts:
    if check_frames(input_folder, "input"):
        break
    attempts += 1
    print(f"Attempt {attempts} to verify frames failed. Retrying in 2 seconds...")
    time.sleep(2)
    if attempts == max_attempts:
        raise RuntimeError("Maximum verification attempts reached for input frames. Cannot proceed.")

# --- 2. Build the NAFNet Model ---
%cd /content/NAFNet
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

opt_path = 'options/test/REDS/NAFNet-width64.yml'
opt = parse(opt_path, is_train=False)
opt['dist'] = False
NAFNet_model = create_model(opt)

# --- 3. Set up Batch Processing & Resumability ---
# Create temporary local folders for faster I/O during inference
temp_input_folder = '/content/NAFNet/temp_deblur_input'
temp_output_folder = '/content/NAFNet/temp_deblur_output'

# Determine where to start processing from
processed_files = os.listdir(output_folder)
start_frame = 0
if processed_files:
    # Extract frame numbers from filenames
    processed_frames = [int(re.findall(r'(\d+)', f)[0]) for f in processed_files if f.endswith('.jpg') and re.findall(r'(\d+)', f)]
    if processed_frames:
        start_frame = max(processed_frames) + 1
print(f"\nResuming deblur process from frame: {start_frame}")

# Get the full list of input files that need to be processed
all_input_files = sorted(glob.glob(os.path.join(input_folder, '*.jpg')))
files_to_process = [f for f in all_input_files if int(re.findall(r'(\d+)', os.path.basename(f))[0]) >= start_frame]

if not files_to_process:
    print("No new frames to process. Deblurring is already complete.")
else:
    # --- 4. The Main Processing Loop ---
    batch_size = 50 # Internal setting for stable processing, not exposed to user.
    total_files = len(files_to_process)

    with tqdm(total=total_files, desc="Deblurring images in batches") as pbar:
        for i in range(0, total_files, batch_size):
            # Ensure temp folders are clean for the new batch
            for folder in [temp_input_folder, temp_output_folder]:
                if os.path.isdir(folder):
                    shutil.rmtree(folder)
                os.makedirs(folder)

            batch_files = files_to_process[i:i+batch_size]

            # Copy current batch to local temp folder for speed
            for file_path in batch_files:
                shutil.copy(file_path, temp_input_folder)

            # Process the batch from the fast local folder
            temp_img_paths = sorted(glob.glob(os.path.join(temp_input_folder, "*.jpg")))
            for img_path in temp_img_paths:
                output_path = os.path.join(temp_output_folder, os.path.basename(img_path))

                img_input = imread(img_path)
                tensor_input = img2tensor(img_input)
                single_image_inference(NAFNet_model, tensor_input, output_path)

            # Copy results from temp folder back to the final destination (e.g., Google Drive)
            processed_batch_files = glob.glob(os.path.join(temp_output_folder, "*.jpg"))
            for result_file in processed_batch_files:
                shutil.copy(result_file, output_folder)

            pbar.update(len(batch_files))

# --- 5. Post-run Check of Output Frames ---
print("\nDeblurring process finished. Verifying all output frames...")
attempts = 0
while attempts < max_attempts:
    if check_frames(output_folder, "output"):
        break
    attempts += 1
    print(f"Attempt {attempts} to verify output frames failed. Retrying in 2 seconds...")
    time.sleep(2)
    if attempts == max_attempts:
        print("Warning: Maximum output verification attempts reached.")

In [ ]:
#@title ##**Create Video** { display-mode: "form" }
import cv2
import os
import subprocess
import time
from tqdm.notebook import tqdm
import torch
import gc

# --- Clean up GPU memory before starting ---
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

# --- User Parameter for Blending ---
deblurred_opacity = 100 #@param {type:"slider", min:0, max:100, step:1}
# 100 = fully deblurred video. 0 = original video. 50 = 50% blend.

# --- Initial Setup ---
if 'file_name' not in locals() or not file_name or not os.path.exists(file_name):
    raise ValueError("file_name is not defined or the file does not exist. Run 'Select Video File' cell.")
if 'input_folder' not in locals() or 'output_folder' not in locals():
    raise NameError("Folder paths are not defined. Please run the 'Config' cell first.")

# ИСПРАВЛЕНО: Определяем директорию для сохранения видео на уровень выше, чем папка с кадрами
save_directory = os.path.dirname(output_folder)
base_file_name = os.path.basename(file_name)
output_file_name = os.path.splitext(base_file_name)[0] + f'_deblurred_{deblurred_opacity}.mp4'
output_file_path = os.path.join(save_directory, output_file_name)
temp_video_path = "/content/temp_video.mp4"

print(f"Original video: {file_name}")
print(f"Final video will be saved to: {output_file_path}")

# --- Get Video Properties ---
cap = cv2.VideoCapture(file_name)
fps_of_video = cap.get(cv2.CAP_PROP_FPS)
cap.release()

# --- Get Frame Lists ---
deblurred_img_files = sorted(glob.glob(os.path.join(output_folder, '*.jpg')))
if not deblurred_img_files:
    raise ValueError(f"No deblurred images found in {output_folder}")

if deblurred_opacity < 100:
    original_img_files = sorted(glob.glob(os.path.join(input_folder, '*.jpg')))
    if not original_img_files:
        raise ValueError(f"No original images found in {input_folder} for blending.")
    if len(deblurred_img_files) != len(original_img_files):
        print(f"Warning: Mismatch in frame count! Deblurred: {len(deblurred_img_files)}, Original: {len(original_img_files)}. Blending may be incorrect.")

# --- Prepare Video Writer ---
first_frame = cv2.imread(deblurred_img_files[0])
height, width, _ = first_frame.shape
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(temp_video_path, fourcc, fps_of_video, (width, height))

# --- Process and Write Frames ---
if deblurred_opacity == 100:
    print("Assembling fully deblurred frames...")
    for img_file in tqdm(deblurred_img_files, desc="Writing frames"):
        frame = cv2.imread(img_file)
        if frame.shape[:2] != (height, width):
            frame = cv2.resize(frame, (width, height))
        writer.write(frame)
else:
    print(f"Blending deblurred frames with original at {deblurred_opacity}% opacity...")
    alpha = deblurred_opacity / 100.0
    beta = 1.0 - alpha
    num_frames_to_process = min(len(deblurred_img_files), len(original_img_files))
    for i in tqdm(range(num_frames_to_process), desc="Blending and writing frames"):
        deblurred_frame = cv2.imread(deblurred_img_files[i])
        original_frame = cv2.imread(original_img_files[i])
        if deblurred_frame.shape[:2] != (height, width):
            deblurred_frame = cv2.resize(deblurred_frame, (width, height))
        if original_frame.shape[:2] != (height, width):
            original_frame = cv2.resize(original_frame, (width, height))
        blended_frame = cv2.addWeighted(deblurred_frame, alpha, original_frame, beta, 0)
        writer.write(blended_frame)

writer.release()
gc.collect()
print("Temporary video file created.")

# --- Encode with FFmpeg for better quality/compression ---
def get_video_bitrate(video_path):
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', video_path]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        return int(result.stdout.strip())
    except (subprocess.CalledProcessError, ValueError):
        return None

bitrate = get_video_bitrate(file_name)
bitrate_str = f'{int(bitrate * 1.2) // 1000}k' if bitrate else '15000k'
print(f"Encoding with target bitrate: {bitrate_str}")

temp_converted_path = "/content/temp_converted.mp4"
cmd = ['ffmpeg', '-i', temp_video_path, '-c:v', 'libx264', '-b:v', bitrate_str, '-pix_fmt', 'yuv420p', '-y', temp_converted_path]
subprocess.run(cmd, check=True, capture_output=True)
os.remove(temp_video_path)
os.rename(temp_converted_path, temp_video_path)
print("Video re-encoded with libx264.")

# --- Mux Audio and Subtitles from Original Video ---
print("Muxing audio/subtitles from original video...")
cmd = ['ffmpeg', '-i', temp_video_path, '-i', file_name, '-map', '0:v', '-map', '1:a?', '-map', '1:s?', '-c', 'copy', '-y', output_file_path]
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("--- FFmpeg Muxing Failed ---")
    print(f"FFmpeg stderr:\n{result.stderr}")
    shutil.copy(temp_video_path, output_file_path)
    print("\nWarning: Could not add original audio/subs. Video-only file was saved.")
else:
    print("Successfully muxed video and audio.")

# --- Final Cleanup ---
if os.path.exists(temp_video_path):
    os.remove(temp_video_path)
if os.path.exists(output_file_path):
    print(f"\nVideo creation complete! File saved at: {output_file_path}")
else:
    print("\nError: Final video file was not created.")

In [ ]:
#@title ##**Download Video** { display-mode: "form" }
import os
import time
from tqdm.notebook import tqdm
from google.colab import files

try:
    if 'output_file_path' in locals() and os.path.exists(output_file_path):
        print(f"Initiating download for: {output_file_path}")

        with tqdm(total=100, desc="Sending file to browser") as pbar:
            files.download(output_file_path)
                        for i in range(100):
                time.sleep(0.02)
                pbar.update(1)

        print("\nDownload should now be active in your browser.")

    else:
        print("Error: Could not find the final video file.")
        print("Please ensure the 'Create Video' cell was executed successfully.")
except NameError:
    print("Error: The variable 'output_file_path' is not defined.")
    print("Please run the 'Create Video' cell first.")